In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd

path = '/kaggle/input/nlp-cs-2025/'
test_df = pd.read_csv(path +'test_without_labels.csv')
submission_df = pd.read_csv(path+'train_submission.csv')

print("Contents of test_without_labels.csv:")
print(test_df.head())

print("\nContents of train_submission.csv:")
print(submission_df.head())

In [ ]:
import pandas as pd
import re
import unicodedata
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import torch
from transformers import EarlyStoppingCallback

# Check if CUDA is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load the data
path = '/kaggle/input/nlp-cs-2025/'
train_df = pd.read_csv(path+'train_submission.csv')
test_df = pd.read_csv(path+'test_without_labels.csv')

def clean_text(text):
    """Clean text while preserving language-specific characteristics"""
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize('NFC', text)

    text = re.sub(r'\s+', ' ', text)

    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    text = ''.join(ch for ch in text if unicodedata.category(ch)[0] != 'C' or ch in ['\n', '\t'])

    return text.strip()

print("Cleaning text data...")
train_df['Text'] = train_df['Text'].apply(clean_text)
test_df['Text'] = test_df['Text'].apply(clean_text)

train_df = train_df.dropna(subset=['Text', 'Label'])
train_df = train_df[train_df['Text'].str.len() > 0]

test_df['Text'] = test_df['Text'].fillna('')
test_df = test_df[test_df['Text'].str.len() > 0]

class_counts = train_df['Label'].value_counts()
print(f"Number of classes: {len(class_counts)}")
print(f"Most common classes: {class_counts.head().to_dict()}")
print(f"Least common classes: {class_counts.tail().to_dict()}")

min_samples = 2
rare_classes = class_counts[class_counts < min_samples].index
train_df = train_df[~train_df['Label'].isin(rare_classes)]
print(f"Number of classes after removing rare classes: {train_df['Label'].nunique()}")

train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df['Label'],
    random_state=42
)

print(f"Training samples: {len(train_data)}, Validation samples: {len(val_data)}")

label_encoder = LabelEncoder()
label_encoder.fit(train_df['Label'])
train_data['label_id'] = label_encoder.transform(train_data['Label'])
val_data['label_id'] = label_encoder.transform(val_data['Label'])

print(f"Number of unique languages: {len(label_encoder.classes_)}")
print(f"Sample languages: {label_encoder.classes_[:10].tolist()}")

# Convert to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(val_data)

# Load the pre-trained model and tokenizer
model_name = "papluca/xlm-roberta-base-language-detection"
print(f"Loading tokenizer and model from {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define tokenization function with adaptive length
def tokenize_function(examples):
    # Calculate token statistics on a subset if needed
    # text_lengths = [len(tokenizer.encode(text)) for text in examples['Text'][:100]]
    # max_observed_length = max(text_lengths)
    # print(f"Max observed token length in sample: {max_observed_length}")

    return tokenizer(
        examples['Text'],
        padding='max_length',
        truncation=True,
        max_length=128,  # Increased from 96 for better language detection
        return_tensors="pt"
    )

# Tokenize the datasets
print("Tokenizing datasets...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

# Rename 'label_id' to 'labels' (required by Trainer)
tokenized_train = tokenized_train.rename_column('label_id', 'labels')
tokenized_val = tokenized_val.rename_column('label_id', 'labels')

# Set format for PyTorch tensors
tokenized_train.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
tokenized_val.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

# Load the model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_encoder.classes_),
    ignore_mismatched_sizes=True  # Fixes classifier size mismatch issue
)

# Move model to the appropriate device
model.to(device)

# Define compute metrics function
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average='macro')
    precision_macro = precision_score(labels, preds, average='macro')
    recall_macro = recall_score(labels, preds, average='macro')

    # Add per-class metrics for the most common classes
    results = {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro
    }

    return results

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=6,  # Increased from 4
    per_device_train_batch_size=32,  # Reduced for more stability
    per_device_eval_batch_size=64,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_dir='./logs',
    logging_steps=50,
    report_to=["none"],  # Disable wandb
    weight_decay=0.01,
    learning_rate=3e-5,  # Slightly reduced for stability
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),  # Enable mixed precision if CUDA is available
    gradient_accumulation_steps=2,  # Accumulate gradients for effectively larger batch
)

# Create Trainer instance with early stopping
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Starting training...")
trainer.train()

print("Evaluating model...")
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

print("Preparing test predictions...")
test_dataset = Dataset.from_pandas(test_df)
tokenized_test = test_dataset.map(tokenize_function, batched=True)
tokenized_test.set_format('torch', columns=['input_ids', 'attention_mask'])

predictions = trainer.predict(tokenized_test)
pred_label_ids = np.argmax(predictions.predictions, axis=1)
pred_labels = label_encoder.inverse_transform(pred_label_ids)

submission = test_df[['Usage', 'Text']].copy()
submission['Label'] = pred_labels

output_path = '/kaggle/working/submission_roberta_enhanced.csv'
submission.to_csv(output_path, index=False)

print(f"Predictions saved to '{output_path}'")

pred_label_counts = pd.Series(pred_labels).value_counts()
print(f"Number of unique predicted languages: {len(pred_label_counts)}")
print(f"Most common predicted languages: {pred_label_counts.head(5).to_dict()}")

In [ ]:
submission.drop(['Text','Usage'], axis=1, inplace=True)
submission['ID'] = submission.index + 1
submission.to_csv('submission_roberta_enhanced.csv', index=False)